# Datavidia 11.0 — Task 1: Klasifikasi `status_penanganan`
**Tim:** _isi_ · **Metrik:** Macro F1 (7 kelas) · **Validasi:** Stratified K-Fold OOF

> Kerangka notebook. Setiap section punya **tujuan**, **checklist**, dan **TODO**. Isi kodenya sendiri.
> Aturan wajib notebook: penjelasan pemrosesan data, penggabungan tabel, pembuatan fitur, pengembangan model, training, inferensi **tanpa retrain** (load file model), dan daftar dependensi.

## Daftar Isi
0. Setup & Konfigurasi
1. Load Data & Optimasi Memori
2. EDA Ringkas
3. Penggabungan Tabel Lookup (1:1)
4. Agregasi Tabel 1:N (scan, rute, invoice)
5. Feature Engineering Lanjutan
6. Encoding Kategori & ID (OOF-safe)
7. Skema Validasi & Baseline
8. Seleksi Fitur
9. Model Utama + Tuning
10. Ensemble
11. Optimasi Threshold Macro F1
12. Analisis Error
13. Simpan Model & Artefak
14. Inferensi Tanpa Retrain → submission.csv
15. Ringkasan & Dependensi

## 0. Setup & Konfigurasi
**Tujuan:** semua yang bisa diubah ada di satu tempat; hasil reproducible.
- [ ] Import library (pandas/polars, numpy, sklearn, lightgbm, catboost, xgboost, optuna, joblib)
- [ ] `SEED` global + set seed numpy/random
- [ ] `DATA_DIR` auto-detect (lokal vs `/kaggle/input/...`)
- [ ] `CFG`: n_folds, seed, daftar kelas (urutan tetap!), path output model
- [ ] Flag `MODE = "train" | "inference"` → section 14 bisa jalan sendiri

In [ ]:
# TODO: imports
# TODO: SEED, set_seed()
# TODO: DATA_DIR auto-detect
# TODO: CFG dict + CLASSES list
# TODO: MODE flag

## 1. Load Data & Optimasi Memori
**Tujuan:** load 9 tabel train & test tanpa kehabisan RAM (scan_events ≈ 470 MB).
- [ ] Fungsi `load_table(name, split)`
- [ ] Downcast numerik (`float64→float32`, `int64→int32/int8`), string kategori → `category`
- [ ] Untuk tabel besar: baca kolom yang perlu saja (`usecols`)
- [ ] Cek: shipment_id unik di shipments; semua shipment test ada di scan_events test
- [ ] Cek: tabel lookup (customers, couriers, hubs, packages, promos) identik train vs test?
- [ ] Print shape + memory usage tiap tabel

In [ ]:
# TODO: load_table(), reduce_mem()
# TODO: sanity checks (unik, coverage, lookup identik)

## 2. EDA Ringkas
**Tujuan:** pahami data, temukan sinyal & jebakan — jangan kebanyakan plot.
- [ ] Distribusi target (imbalanced: normal 38% … klaim_hilang 3%)
- [ ] Missing value per kolom (volume_l ±3%) — apakah missing-nya informatif per kelas?
- [ ] Distribusi tiap fitur numerik **train vs test** (adversarial check di Section 7)
- [ ] Crosstab kategori × target (normalize per baris) — cari kolom yang ada sinyalnya
- [ ] Tabel 1:N: jumlah baris per shipment, event_type vs target, invoice_type vs target
- [ ] Catat hipotesis per kelas (mis. `klaim` invoice → klaim_hilang? banyak `upaya_antar` → gagal_antar/konfirmasi_alamat?)
> Catatan awal: `upaya_antar` lebih sering di konfirmasi_alamat; ada 224k invoice tipe `klaim`. Data disebut punya kolom tidak berguna → curigai kolom noise.

In [ ]:
# TODO: target distribution
# TODO: missing analysis
# TODO: crosstab & hipotesis

## 3. Penggabungan Tabel Lookup (1:1)
**Tujuan:** shipments + customers + couriers + hubs + service_packages + promotions.
- [ ] Merge `left` pakai ID; **assert jumlah baris tidak berubah** setelah tiap merge
- [ ] Prefix nama kolom per tabel (`cust_`, `cour_`, `hub_`, `pkg_`, `promo_`) biar nggak bentrok
- [ ] `TANPA_PROMO` → tangani sebagai kategori sendiri
- [ ] Fitur turunan relasi: `hub` shipment == `hub` kurir? region customer == dest_region?

In [ ]:
# TODO: merge bertahap + assert len
# TODO: fitur relasi antar tabel

## 4. Agregasi Tabel 1:N
**Tujuan:** ringkas log jadi 1 baris/shipment. Ini kemungkinan sumber skor terbesar.
### 4a. scan_events
- [ ] count total, count per `event_type` (pivot), proporsi tiap event
- [ ] `gap_hours`: sum / mean / max / std, gap per event_type
- [ ] `is_manual` rate, `scan_quality` min/mean, count per `device_type`
- [ ] Urutan: event pertama/terakhir, jumlah `upaya_antar`, ada `selesai`?, jumlah hub unik
- [ ] (opsional) urutan event sebagai string → frequency encode
### 4b. route_segments
- [ ] jumlah kaki rute, moda unik, ada laut/udara, sum/max durasi, jarak, biaya
- [ ] kecepatan rata-rata (jarak/durasi), biaya per km, beda total jarak segmen vs `distance_km`
### 4c. invoices
- [ ] count per `invoice_type` (terutama `klaim`), sum amount/discount/tax per tipe
- [ ] `is_penalty` sum, `payment_status` count, `days_to_pay` mean/max
- [ ] Hati-hati: invoice `langganan` level customer → cek apakah per-shipment atau per-customer
- [ ] Join semua agregat → **assert jumlah baris tetap & tidak ada duplikasi**

In [ ]:
# TODO: agg_scan(df) -> DataFrame index shipment_id
# TODO: agg_route(df)
# TODO: agg_invoice(df)
# TODO: join + assert

## 5. Feature Engineering Lanjutan
- [ ] Rasio: declared_value/weight, berat_volumetrik/weight, volume/weight, biaya/declared_value
- [ ] Waktu: total durasi rute vs `estimasi_hari`×24 (indikasi telat), ship_hour vs jam buka/tutup hub
- [ ] Interaksi domain: fragility_flag × fragile_score × moda laut; address_quality × flag_alamat_baru × jumlah_catatan
- [ ] Frequency count: courier, hub, customer, kota_asal, kota_tujuan, rute (asal→tujuan)
- [ ] Statistik grup **tanpa target** (aman): rata-rata gap_hours per hub, deviasi shipment dari rata-rata hubnya
- [ ] Setiap fitur baru: catat hipotesis + uji delta CV (jangan tumpuk tanpa bukti)

In [ ]:
# TODO: ratio features
# TODO: time/delay features
# TODO: frequency & group stats

## 6. Encoding Kategori & ID (OOF-safe) ⚠️ titik rawan leakage
- [ ] Kategori kardinalitas rendah → native categorical LightGBM/CatBoost
- [ ] ID kardinalitas tinggi (courier, hub, customer, kota) → **target encoding multi-kelas OOF**:
      untuk tiap fold, hitung rate per kelas dari fold train saja, terapkan ke fold valid
- [ ] Smoothing (prior global, m ≈ 20–100) supaya ID jarang nggak overfit
- [ ] Test: encoding dari **seluruh** train
- [ ] TE harus dihitung **di dalam** loop CV model (bukan sekali di depan!), kalau tidak skor CV bocor

In [ ]:
# TODO: oof_target_encode(train, test, col, folds, smoothing)

## 7. Skema Validasi & Baseline
**Tujuan:** CV yang bisa dipercaya > skor public LB (public cuma 30%, ±0.003 noise).
- [ ] `StratifiedKFold(5, shuffle, SEED)` — simpan fold id ke kolom, pakai sama untuk semua model
- [ ] Baseline 1: tebak kelas mayoritas → Macro F1 (batas bawah)
- [ ] Baseline 2: LightGBM fitur shipments saja
- [ ] **Adversarial validation**: klasifikasi train vs test; AUC ≈ 0.5 → split acak, aman pakai StratifiedKFold
- [ ] Laporkan mean ± std F1 antar fold
- [ ] Bandingkan CV vs public LB tiap submit → kalau tidak sejalan, ada yang bocor

In [ ]:
# TODO: folds
# TODO: baselines
# TODO: adversarial validation
# TODO: eval helper: macro_f1(y, proba, weights=None)

## 8. Seleksi Fitur (buang kolom noise)
- [ ] **Null importance**: latih ulang dengan target di-shuffle N kali; buang fitur yang importance aslinya ≤ distribusi null
- [ ] Atau permutation importance pada OOF
- [ ] Cek korelasi tinggi/duplikat (>0.98) → simpan salah satu
- [ ] Bandingkan CV sebelum/sesudah — terima hanya kalau naik/stabil

In [ ]:
# TODO: null importance
# TODO: final FEATURES list

## 9. Model Utama + Tuning
- [ ] LightGBM multiclass: early stopping pada fold valid, `class_weight` atau tanpa (bandingkan)
- [ ] CatBoost (native cat features)
- [ ] XGBoost (opsional, untuk diversitas)
- [ ] Optuna: tuning di **subset fold/sampel**, budget terbatas; jangan tuning berlebihan ke CV (overfit ke CV juga bisa)
- [ ] Simpan OOF proba (n_train×7) & test proba (rata-rata antar fold) per model
- [ ] Multi-seed averaging untuk model final

In [ ]:
# TODO: train_lgb(folds) -> oof, test_pred, models
# TODO: train_cat(...)
# TODO: optuna (opsional)

## 10. Ensemble
- [ ] Rata-rata proba sederhana dulu
- [ ] Weight optimization pada OOF (scipy minimize log loss) — pakai sedikit parameter saja
- [ ] (opsional) Stacking: logistic regression di atas OOF proba, dengan CV terpisah
- [ ] Keep it simple: ensemble kompleks yang cuma +0.0005 di CV = noise

In [ ]:
# TODO: blend OOF, evaluasi macro F1

## 11. Optimasi Threshold Macro F1
**Ide:** argmax proba ≠ optimal untuk Macro F1. Kalikan proba dengan bobot per kelas `w_k`, lalu argmax.
- [ ] Coordinate ascent / grid per kelas di OOF untuk cari `w` (7 angka)
- [ ] **Anti-overfit:** fit `w` di fold A, evaluasi di fold B (nested), atau pastikan `w` stabil antar fold
- [ ] Bandingkan distribusi kelas prediksi vs distribusi asli
- [ ] Terapkan `w` yang sama ke test proba

In [ ]:
# TODO: optimize_class_weights(oof_proba, y)
# TODO: validasi stabilitas antar fold

## 12. Analisis Error
- [ ] Confusion matrix (normalize per baris) — kelas mana yang saling ketuker?
- [ ] F1 per kelas — kelas terlemah jadi fokus fitur berikutnya
- [ ] Feature importance (gain) / SHAP ringkas untuk 2–3 kelas jarang

In [ ]:
# TODO: confusion matrix, classification_report, SHAP

## 13. Simpan Model & Artefak
- [ ] Model per fold: LightGBM `.txt`, CatBoost `.cbm`, XGBoost `.json`
- [ ] Encoder TE, daftar fitur, bobot ensemble, bobot kelas → `.joblib`/`.json`
- [ ] Versi library → `requirements.txt`

In [ ]:
# TODO: save_artifacts(dir)

## 14. Inferensi Tanpa Retrain
**Wajib:** jalan dari nol dengan `MODE="inference"` — load artefak, bangun fitur test, prediksi.
- [ ] Pipeline fitur test = fungsi yang sama dengan train (jangan copy-paste beda)
- [ ] Load model → proba → blend → bobot kelas → argmax → label
- [ ] Validasi: 362.245 baris, urutan sama `sample_submission`, label ∈ 7 kelas
- [ ] Cek reproducibility: hasil inference == hasil saat training (diff = 0)

In [ ]:
# TODO: inference pipeline -> submission.csv

## 15. Ringkasan & Dependensi
- [ ] Tabel eksperimen: versi | fitur | model | CV F1 ± std | public LB
- [ ] 2 final pick: (1) CV terbaik, (2) hedge yang lebih sederhana/robust
- [ ] Dependensi + langkah menjalankan inference
- [ ] Sumber & versi model eksternal (jika ada, mis. TabPFN)

### Anti-overfit checklist
- [ ] Semua yang pakai target (TE, threshold, stacking) dihitung di dalam fold
- [ ] Keputusan berdasarkan CV mean ± std, bukan public LB
- [ ] Peningkatan < std antar fold = anggap tidak ada
- [ ] Tidak ada tuning berlebih ke public LB